# 2.2.Подобрать источники данных по требованиям п.3 и изучить их. Выбрать числовые и категориальные характеристики, представляющие аналитический портрет состояния объекта исследования и факторов влияния.

## 3.Среди открытых источников данных по теме исследования найти:

### 3.1.не менее 1 реляционного набора данных;

In [1]:
import time
import pandas as pd

cols = ["loan_amnt", "term", "int_rate", "installment", "grade", "emp_length",
        "home_ownership", "annual_inc", "verification_status", "issue_d",
        "loan_status", "purpose", "addr_state", "dti", "delinq_2yrs",
        "earliest_cr_line", "fico_range_low", "fico_range_high",
        "open_acc", "pub_rec", "revol_bal", "revol_util"]

t0 = time.perf_counter()
lc = pd.read_csv("accepted_2007_to_2018Q4.csv", usecols=cols, low_memory=False)
lc = lc.dropna(subset=["loan_status"])          # служебные строки в конце файла
print(f"Загрузка: {time.perf_counter() - t0:.1f} c")

print("Размер:", lc.shape)
print("\nТипы:\n", lc.dtypes.value_counts())
print("\nПропуски, %:\n", (lc.isna().mean() * 100).round(2).sort_values(ascending=False).head(8))

lc["issue_month"] = pd.to_datetime(lc["issue_d"], format="%b-%Y")
print("\nПериод выдачи:", lc["issue_month"].min().date(), "-", lc["issue_month"].max().date())

print("\nСтатусы кредитов, %:\n", (lc["loan_status"].value_counts(normalize=True) * 100).round(1))

done = lc[lc["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])]
print(f"\nЗавершённых кредитов: {len(done):,}; доля дефолтов: "
      f"{done['loan_status'].isin(['Charged Off', 'Default']).mean():.1%}")

lc.sample(3, random_state=1)

Загрузка: 16.2 c
Размер: (2260668, 22)

Типы:
 float64    12
object     10
Name: count, dtype: int64

Пропуски, %:
 emp_length         6.50
revol_util         0.08
dti                0.08
addr_state         0.00
revol_bal          0.00
pub_rec            0.00
open_acc           0.00
fico_range_high    0.00
dtype: float64

Период выдачи: 2007-06-01 - 2018-12-01

Статусы кредитов, %:
 loan_status
Fully Paid                                             47.6
Current                                                38.9
Charged Off                                            11.9
Late (31-120 days)                                      0.9
In Grace Period                                         0.4
Late (16-30 days)                                       0.2
Does not meet the credit policy. Status:Fully Paid      0.1
Does not meet the credit policy. Status:Charged Off     0.0
Default                                                 0.0
Name: proportion, dtype: float64

Завершённых кредитов: 1,345,

,loan_amnt,term,int_rate,installment,grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,...,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,open_acc,pub_rec,revol_bal,revol_util,issue_month
1832791,35000.0,60 months,21.98,966.27,E,9 years,RENT,140000.0,Verified,Jun-2013,...,16.55,1.0,Jun-1996,680.0,684.0,25.0,0.0,35106.0,25.2,2013-06-01
1380750,30000.0,36 months,6.46,918.93,A,2 years,RENT,77000.0,Source Verified,Dec-2018,...,14.03,0.0,May-1989,720.0,724.0,5.0,0.0,17643.0,60.8,2018-12-01
560754,13600.0,36 months,12.62,455.76,C,3 years,OWN,40000.0,Source Verified,Sep-2017,...,20.76,3.0,Apr-2003,665.0,669.0,20.0,0.0,7417.0,72.7,2017-09-01


Вывод. Набор содержит 2 260 668 кредитов, выданных с июня 2007 по декабрь 2018 года, по 22 отобранным признакам (12 числовых и 10 категориальных или текстовых). Загрузка заняла 16,2 с. Требования п. 3 выполнены: категориальных признаков больше трёх (grade, purpose, home_ownership, addr_state, term, verification_status), числовых больше трёх (loan_amnt, int_rate, dti, annual_inc, fico_range_low), есть временная характеристика (issue_d).

Исход кредита известен только для завершённых кредитов (Fully Paid, Charged Off, Default). Таких 1 345 350, то есть 59,5 % набора. Остальные кредиты (в основном Current, 38,9 %) исключаются из обучающей выборки, так как их исход ещё не наблюдается (цензурирование). Доля дефолтов среди завершённых кредитов составляет 20,0 % (около 269 тыс. кредитов), то есть классы несбалансированы, что необходимо учесть при моделировании.

Пропусков в отобранных признаках мало: больше всего в emp_length (6,5 %), в dti и revol_util по 0,08 %. Эти пропуски обработаем на этапе подготовки данных.

Общие атрибуты для слияния с другими источниками: addr_state (штат) и issue_month (месяц выдачи).

### 3.2.не менее 1 иерархического набора данных (формат XML);

In [ ]:
import os, time
import requests
import xml.etree.ElementTree as ET
import pandas as pd

API_KEY = ""          # не публикуйте ключ в репозитории
BASE = "https://api.stlouisfed.org/fred"
os.makedirs("data/raw/xml", exist_ok=True)

STATES = ["AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID","IL",
          "IN","IA","KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE",
          "NV","NH","NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI","SC","SD",
          "TN","TX","UT","VT","VA","WA","WV","WI","WY"]

def get_xml(endpoint, **params):
    params.update(api_key=API_KEY, file_type="xml")
    r = requests.get(f"{BASE}/{endpoint}", params=params, timeout=30)
    r.raise_for_status()
    return r.content

rows, meta = [], []
t0 = time.perf_counter()
for st in STATES:
    sid = f"{st}UR"
    raw = get_xml("series/observations", series_id=sid,
                  observation_start="2006-01-01", observation_end="2019-12-31")
    with open(f"data/raw/xml/{sid}.xml", "wb") as f:
        f.write(raw)
    for o in ET.fromstring(raw).findall("observation"):
        v = o.get("value")
        rows.append({"state": st, "date": o.get("date"),
                     "unemployment": None if v == "." else float(v)})

    s = ET.fromstring(get_xml("series", series_id=sid)).find("series")
    meta.append({"state": st, "title": s.get("title"), "frequency": s.get("frequency"),
                 "units": s.get("units"), "seasonal_adjustment": s.get("seasonal_adjustment")})
    time.sleep(0.6)                 # лимит API: 120 запросов в минуту
print(f"Загрузка и разбор XML: {time.perf_counter() - t0:.1f} c")

# структура иерархии на примере одного файла
root = ET.parse("data/raw/xml/CAUR.xml").getroot()
print("Корень:", root.tag, "| атрибутов:", len(root.attrib), "| count =", root.get("count"))
print("Первый потомок:", root[0].tag, root[0].attrib)

df_unemp = pd.DataFrame(rows)
df_unemp["date"] = pd.to_datetime(df_unemp["date"])
df_meta = pd.DataFrame(meta)

print("\nРазмер:", df_unemp.shape, "| штатов:", df_unemp["state"].nunique())
print("Период:", df_unemp["date"].min().date(), "-", df_unemp["date"].max().date())
print("Пропуски:", df_unemp["unemployment"].isna().sum())
print(df_unemp["unemployment"].describe().round(2))
df_meta.head(3)

Загрузка и разбор XML: 99.7 c
Корень: observations | атрибутов: 12 | count = 168
Первый потомок: observation {'realtime_start': '2026-10-23', 'realtime_end': '2026-10-23', 'date': '2006-01-01', 'value': '5.000'}

Размер: (8568, 3) | штатов: 51
Период: 2006-01-01 - 2019-12-01
Пропуски: 0
count    8568.00
mean        5.74
std         2.24
min         2.00
25%         4.00
50%         5.20
75%         7.10
max        14.30
Name: unemployment, dtype: float64


,state,title,frequency,units,seasonal_adjustment
0,AL,Unemployment Rate in Alabama,Monthly,Percent,Seasonally Adjusted
1,AK,Unemployment Rate in Alaska,Monthly,Percent,Seasonally Adjusted
2,AZ,Unemployment Rate in Arizona,Monthly,Percent,Seasonally Adjusted


Вывод. Загружено 8 568 наблюдений по 51 штату за период с января 2006 по декабрь 2019 года, то есть по 168 месяцев на штат (51 × 168 = 8 568). Пропусков нет. Загрузка и разбор заняли 99,7 с на 102 запроса (наблюдения и метаданные по каждому штату). Около 31 с из них составляют намеренные паузы 0,6 с между штатами из-за лимита API в 120 запросов в минуту.

Иерархия файла: корневой элемент observations с 12 атрибутами запроса (период, единицы, сортировка, количество записей count = 168) и 168 вложенных элементов observation с атрибутами date и value. Пропущенное значение в XML кодируется точкой, но в данной выгрузке такие значения не встретились. Сырые файлы сохранены в data/raw/xml/.

Уровень безработицы по штатам и месяцам лежит в диапазоне от 2,0 % до 14,3 % при среднем 5,74 % и стандартном отклонении 2,24 п.п. Квартили 4,0 %, 5,2 % и 7,1 % показывают, что распределение смещено вправо. Это отражает пик кризиса 2009–2010 годов и существенные различия между штатами.

Состав характеристик:

Числовые: уровень безработицы в процентах (шкала отношений). Производные признаки на этапе подготовки: изменение за 3, 6 и 12 месяцев, скользящее среднее, лаг.
Категориальные: штат, название серии, частота (месячная), единицы измерения (проценты), тип сезонной корректировки (Seasonally Adjusted). Три последних признака постоянны для всех штатов, поэтому на этапе подготовки добавим производные признаки: регион переписи США и признак периода рецессии.
Время: date, месячная частота.

Охват 51 штат позволяет строить географический срез и проверять гипотезу H1.4 (связь безработицы со штатами и долей дефолтов). Ключи слияния с Lending Club: state ↔ addr_state и date ↔ issue_month.

### 3.3.не менее 1 источника с API с выдачей в формате JSON;

Датасет: FRED API, эффективная ставка по федеральным фондам (FEDFUNDS), file_type=json. Это макроэкономический фактор стоимости денег: ставка ФРС влияет на ставки по потребительским кредитам и на долговую нагрузку заёмщиков. Ответ API содержит параметры запроса на верхнем уровне и список наблюдений во вложенном ключе observations.

In [3]:
import json, time
import pandas as pd

t0 = time.perf_counter()
with open("FEDFUNDS.json", encoding="utf-8") as f:
    raw = json.load(f)
print(f"Чтение и разбор JSON: {time.perf_counter() - t0:.4f} c")

print("Ключи верхнего уровня:", list(raw.keys()))
print("Наблюдений в ответе (count):", raw["count"])
print("Пример записи:", raw["observations"][0])

ff = pd.json_normalize(raw["observations"])[["date", "value"]]
ff["date"] = pd.to_datetime(ff["date"])
ff["value"] = pd.to_numeric(ff["value"], errors="coerce")   # "." -> NaN
ff = ff.rename(columns={"value": "fed_funds_rate"})
print("Полный ряд:", ff["date"].min().date(), "-", ff["date"].max().date(), "| пропусков:", ff["fed_funds_rate"].isna().sum())

# общий период проекта; начало с 2005 для расчёта лагов
ff = ff[(ff["date"] >= "2005-01-01") & (ff["date"] <= "2019-12-01")].reset_index(drop=True)
ff["chg_3m"] = ff["fed_funds_rate"].diff(3)
ff["chg_12m"] = ff["fed_funds_rate"].diff(12)
ff["ma_6m"] = ff["fed_funds_rate"].rolling(6).mean()

print("\nПериод анализа:", ff["date"].min().date(), "-", ff["date"].max().date(), "| строк:", len(ff))
print(ff["fed_funds_rate"].describe().round(2))
ff.tail(3)

Чтение и разбор JSON: 0.0030 c
Ключи верхнего уровня: ['realtime_start', 'realtime_end', 'observation_start', 'observation_end', 'units', 'output_type', 'file_type', 'order_by', 'sort_order', 'count', 'offset', 'limit', 'observations']
Наблюдений в ответе (count): 867
Пример записи: {'realtime_start': '2026-10-21', 'realtime_end': '2026-10-21', 'date': '1954-07-01', 'value': '0.80'}
Полный ряд: 1954-07-01 - 2026-09-01 | пропусков: 0

Период анализа: 2005-01-01 - 2019-12-01 | строк: 180
count    180.00
mean       1.43
std        1.73
min        0.07
25%        0.13
50%        0.38
75%        2.30
max        5.26
Name: fed_funds_rate, dtype: float64


,date,fed_funds_rate,chg_3m,chg_12m,ma_6m
177,2019-10-01,1.83,-0.57,-0.36,2.195000
178,2019-11-01,1.55,-0.58,-0.65,2.055000
179,2019-12-01,1.55,-0.49,-0.72,1.916667


Вывод (подставьте цифры из запуска). Из JSON-ответа извлечено 867 месячных наблюдений за период с июля 1954 года, пропусков [число]. Разбор занял [t] с. Структура ответа: метаданные запроса на верхнем уровне (период, единицы, сортировка, count) и список словарей observations с полями date и value, причём значения приходят строками и требуют приведения к числу. Для проекта оставлен период с января 2005 по декабрь 2019 года ([N] месяцев): год запаса нужен для расчёта изменений за 12 месяцев, а анализ ведётся с 2006 года. Ставка в этот период менялась от [min] % до [max] % при среднем [mean] %: почти нулевой уровень после кризиса 2008 года и постепенное повышение в 2016–2018 годах.

Состав характеристик:

Числовые: значение ставки в процентах (шкала отношений). Производные признаки: изменение за 3 и 12 месяцев, скользящее среднее за 6 месяцев.
Время: date, месячная частота.
Категориальные: собственных категориальных полей в ответе нет, они есть только в метаданных серии (частота, единицы, сезонная корректировка). Для разнообразия добавим на этапе подготовки производный признак режима ставки (например, «около нуля», «рост», «снижение») и признак периода рецессии.

Ограничение: одна серия слабо покрывает требование «не менее трёх числовых и трёх категориальных характеристик» и факторную полноту источника. Для усиления в этот же JSON-источник стоит добавить ещё три серии через тот же API: UMCSENT (потребительские настроения), CPIAUCSL (инфляция) и DRCCLACBS (просрочка по кредитным картам). Код загрузки у меня уже есть, он занимает 4 запроса. Решите, берём ли мы все четыре серии или оставляем только FEDFUNDS.

Ключ слияния с Lending Club: date ↔ issue_month, без привязки к штату, так как ставка единая для США. Чтобы не использовать информацию из будущего, ставка берётся на месяц выдачи кредита или с лагом.

### 3.4.не менее 1 источника неструктурированных данных: в текстовом виде (статьи, посты, сообщения, сайты), аудио или видео. Допустимо получение по API.

In [5]:
import time
import pandas as pd

cols = ["Date received", "Product", "Sub-product", "Issue", "Sub-issue",
        "Consumer complaint narrative", "Company", "State", "Submitted via",
        "Date sent to company", "Company response to consumer", "Timely response?", "Complaint ID"]

t0 = time.perf_counter()
total, parts = 0, []
for ch in pd.read_csv("CCDB_Export_1_December_2011_through_April_2018.csv",
                      usecols=cols, chunksize=200_000, low_memory=False):
    total += len(ch)
    ch = ch.dropna(subset=["Consumer complaint narrative"])
    parts.append(ch)
df_txt = pd.concat(parts, ignore_index=True)
print(f"Чтение: {time.perf_counter() - t0:.1f} c")

df_txt["Date received"] = pd.to_datetime(df_txt["Date received"], errors="coerce")
print(f"Всего жалоб в файле: {total:,}; с текстом: {len(df_txt):,} ({len(df_txt)/total:.1%})")
print("Период с текстом:", df_txt["Date received"].min().date(), "-", df_txt["Date received"].max().date())

df_txt["n_words"] = df_txt["Consumer complaint narrative"].str.split().str.len()
print("Штатов:", df_txt["State"].nunique())
print("\nПропуски, %:\n", (df_txt.isna().mean() * 100).round(1).sort_values(ascending=False).head(5))
print("\nТоп-5 продуктов:\n", df_txt["Product"].value_counts().head(5))
print("\nДлина текста, слов:\n", df_txt["n_words"].describe().round(0))
print("\nПример:\n", df_txt["Consumer complaint narrative"].iloc[0][:300])

Чтение: 9.2 c
Всего жалоб в файле: 1,031,863; с текстом: 291,574 (28.3%)
Период с текстом: 2015-03-19 - 2018-04-30
Штатов: 63

Пропуски, %:
 Sub-issue        34.6
Sub-product      17.9
State             0.4
Date received     0.0
Product           0.0
dtype: float64

Топ-5 продуктов:
 Product
Debt collection                                                                 66631
Credit reporting, credit repair services, or other personal consumer reports    53930
Mortgage                                                                        45248
Credit reporting                                                                31587
Credit card                                                                     18837
Name: count, dtype: int64

Длина текста, слов:
 count    291574.0
mean        196.0
std         204.0
min           1.0
25%          72.0
50%         137.0
75%         252.0
max        6314.0
Name: n_words, dtype: float64

Пример:
 I have tried unsuccessfully to contact Citize

Вывод. Файл содержит 1 031 863 жалобы за период с декабря 2011 по апрель 2018 года. Текст есть у 291 574 жалоб (28,3 %), то есть более чем у четверти записей. Чтение файла частями заняло 9,2 с. Тексты с 19 марта 2015 по 30 апреля 2018 года укладываются в общий период проекта.

Тексты свободной формы и очень разной длины: в среднем 196 слов, медиана 137, 75 % жалоб короче 252 слов, максимум 6 314 слов. Среднее заметно выше медианы, то есть распределение сильно скошено вправо из-за редких очень длинных жалоб. На этапе 4 это нужно учесть (ограничение длины или логарифм). Жалобы с текстом распределены по продуктам неравномерно: взыскание долгов составляет 22,9 % (66 631), ипотека 15,5 % (45 248), кредитные карты 6,5 % (18 837). Кредитный рейтинг присутствует под двумя названиями (53 930 и 31 587 жалоб, вместе 29,3 %): это один продукт, который CFPB переименовал, поэтому названия нужно объединить при подготовке данных.

Состав характеристик:

Категориальные: продукт, подпродукт, тема и подтема проблемы, штат, компания, канал подачи, ответ компании, своевременность ответа.
Числовые (извлекаются на этапе 4): длина текста в словах и символах, тональность, срок реакции компании в днях, число жалоб за месяц.
Время: Date received, Date sent to company.

Ограничения и особенности данных:

Текст публикуется только по согласию потребителя и только с марта 2015 года, поэтому общий период проекта: март 2015 – апрель 2018 (38 месяцев).
В поле State 63 значения вместо 51: среди них территории и военные адреса. Для слияния оставляем только 50 штатов и округ Колумбия. Пропусков в штате 0,4 %. Также много пропусков в Sub-issue (34,6 %) и Sub-product (17,9 %), эти поля второстепенные.
Ипотечные жалобы (15,5 % текстов) выходят за границы исследования потребительских кредитов, на этапе подготовки их отфильтруем или оставим как контрольную группу.
Тексты содержат маски персональных данных (XXXX), которые удаляются при очистке.
Жалобы описывают состояние штата, а не конкретного заёмщика Lending Club, поэтому связь с дефолтом оценивается на уровне «штат × месяц», то есть как ассоциация.
Источник загружен файлом экспорта, а не через API: обращения к API закончились ошибками доступа (403 и 404). Это можно указать в записке как причину выбора файла.

Ключи слияния: State ↔ addr_state и месяц Date received ↔ issue_month.